# Intention grounding: stage-by-stage walkthrough

Runs the framework of Figure 2 on the Figure 1 fixture (four vaccine tables) and on the IGBench corpus. Offline: rule-based Interpreter and expression-index Grounder. Set `USE_LLM = True` (with a key in `intention_grounding/.env`) to use the LLM components.

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / 'tests')]
from intention_grounding import il, profiler, aligner
from intention_grounding.interpreter import interpret, RuleInterpreter, LLMInterpreter
from intention_grounding.grounder import ground, IndexResolver, LLMResolver
from intention_grounding.context import ContextMatcher
from intention_grounding.pipeline import Pipeline
from common import fig1_manifest
USE_LLM = False
L = il.load()
INTERP = LLMInterpreter(allow_live=True) if USE_LLM else RuleInterpreter()
RESOLVER = LLMResolver(allow_live=True) if USE_LLM else IndexResolver(L)

## 1. Interpreter: $(q_u, \iota_u)$

In [ ]:
u = 'Compare mild side effect rates across countries in Europe'
r = interpret(u, INTERP, L['vocabulary']).to_dict()
print(r['status'], '| q_u =', r['context_query'])
print(json.dumps(r['intention'], indent=1))

## 2. Grounder: $(g_u, \Psi_u, s_G)$ from the Intention Library

In [ ]:
g = ground(r, L, RESOLVER)
print(g['status'], g['g_u']['variant_id'])
for p in g['psi_u']:
    print(p['procedure_id'], p['expression'])
    for req in p['hard']:
        print('   ', req['requirement_id'], req['axis'], req['predicate'], req['params'])

## 3–5. Profiles $\Phi(T)$ and alignment $A(T,\iota_u)$ on the Figure 1 tables

In [ ]:
M = fig1_manifest()
for t in M['tables']:
    p = profiler.profile(t)
    a = aligner.assess(t, p, g)
    best = next(x for x in a['procedures'] if x['procedure_id'] == a['best_procedure'])
    print(f"{t['table_id']:28s} {a['state']:10s} A={a['A']:.2f} ", ''.join(v['verdict'][0].upper() for v in best['requirements']))

## 6. The same topic, a different intention

In [ ]:
pipe = Pipeline(M, library=L, interpreter=INTERP, resolver=RESOLVER, matcher=ContextMatcher(M, 'bm25', cache_dir=None), k_candidates=4)
for u in ['Compare COVID-19 vaccine mild side effect rates across countries in Europe',
          'Predict the side effect type of a COVID-19 vaccine patient from age, dose and prior condition']:
    rep = pipe.run(u)
    print(u)
    for x in rep['ranking']:
        print(f"  {x['rank']}. {x['table_id']:28s} U={x['U']:.3f} r={x['r']:.3f} {x['state']}")
    print('  timing (ms):', rep['timing'])

## 7. On the IGBench corpus (needs `python -m intention_grounding.corpus build`)

In [ ]:
from intention_grounding import corpus
from intention_grounding.cli import explain_text
MANIFEST = corpus.load_manifest()
big = Pipeline(MANIFEST, library=L, interpreter=INTERP, resolver=RESOLVER, context_method='bm25')
print(explain_text(big.run('Compare the average survival rate of Titanic passengers across passenger classes'), 8))